# Three-Bar Reversal on SPY
## Strategy Brief
The Three-Bar Reversal is a price action pattern used to identify potential reversals in the market. It consists of three consecutive bars where the first and third bars are in the opposite direction, and the third bar closes beyond the midpoint of the first bar. This pattern is used to predict a reversal in the current trend. The strategy involves entering a trade in the direction of the third bar, with the expectation that the reversal will continue.
## References
- https://www.finanztip.de/kfz-versicherung/fuer-rentner/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
Define the trading parameters and constants used in the strategy.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SYMBOL = 'SPY'
THREE_BAR_LOOKBACK = 3

### PHASE 2 - Data Exploration
Download SPY data from Yahoo Finance, compute the necessary indicators, and visualize the data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(SYMBOL, start=START_DATE, end=END_DATE)

# Compute Three-Bar Reversal indicators
data['Three_Bar_High'] = data['High'].shift(THREE_BAR_LOOKBACK)
data['Three_Bar_Low'] = data['Low'].shift(THREE_BAR_LOOKBACK)
data['Three_Bar_Close'] = data['Close'].shift(THREE_BAR_LOOKBACK)

# Plot price and indicators
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.title('SPY Price with Three-Bar Reversal Indicators')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
Define the signal generation logic and trading rules based on the Three-Bar Reversal pattern.

In [ ]:
# Generate signals
data['Signal'] = 0
bullish_condition = (data['Close'] > data['Three_Bar_High']) & (data['Close'].shift(1) < data['Three_Bar_Low'].shift(1))
bearish_condition = (data['Close'] < data['Three_Bar_Low']) & (data['Close'].shift(1) > data['Three_Bar_High'].shift(1))
data.loc[bullish_condition, 'Signal'] = 1
data.loc[bearish_condition, 'Signal'] = -1

# Entry/Exit logic
data['Position'] = data['Signal'].replace(0, np.nan).ffill().fillna(0)

### PHASE 4 - Coding & Backtesting
Implement the backtesting logic to evaluate the strategy's performance.

In [ ]:
# Calculate daily returns
data['Market_Returns'] = data['Close'].pct_change()
data['Strategy_Returns'] = data['Market_Returns'] * data['Position'].shift(1)

data['Equity_Curve'] = (1 + data['Strategy_Returns']).cumprod()
data['Equity_Curve_Market'] = (1 + data['Market_Returns']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.plot(data['Equity_Curve_Market'], label='Market Equity Curve')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
Evaluate the strategy's performance using key metrics and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve):
    total_return = equity_curve.iloc[-1] - 1
    cagr = (equity_curve.iloc[-1]) ** (1 / (len(equity_curve) / 252)) - 1
    daily_returns = equity_curve.pct_change().dropna()
    sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)
    downside_returns = daily_returns[daily_returns < 0]
    sortino_ratio = np.mean(daily_returns) / np.std(downside_returns) * np.sqrt(252)
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return total_return, cagr, sharpe_ratio, sortino_ratio, max_drawdown, calmar_ratio

strategy_metrics = calculate_performance_metrics(data['Equity_Curve'])
market_metrics = calculate_performance_metrics(data['Equity_Curve_Market'])

performance_df = pd.DataFrame({
    'Metric': ['Total Return', 'CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown', 'Calmar Ratio'],
    'Strategy': strategy_metrics,
    'Buy & Hold': market_metrics
})
print(performance_df)

### PHASE 6 - Deploy & Monitor
Create a function to download recent data and compute today's trading signal.

In [ ]:
def get_latest_signal():
    recent_data = yf.download(SYMBOL, period='60d')
    recent_data['Three_Bar_High'] = recent_data['High'].shift(THREE_BAR_LOOKBACK)
    recent_data['Three_Bar_Low'] = recent_data['Low'].shift(THREE_BAR_LOOKBACK)
    recent_data['Three_Bar_Close'] = recent_data['Close'].shift(THREE_BAR_LOOKBACK)
    
    recent_data['Signal'] = 0
    bullish_condition = (recent_data['Close'] > recent_data['Three_Bar_High']) & (recent_data['Close'].shift(1) < recent_data['Three_Bar_Low'].shift(1))
    bearish_condition = (recent_data['Close'] < recent_data['Three_Bar_Low']) & (recent_data['Close'].shift(1) > recent_data['Three_Bar_High'].shift(1))
    recent_data.loc[bullish_condition, 'Signal'] = 1
    recent_data.loc[bearish_condition, 'Signal'] = -1

    today_signal = recent_data['Signal'].iloc[-1]
    print(f"Today's Signal: {'Buy' if today_signal == 1 else 'Sell' if today_signal == -1 else 'Hold'}")

get_latest_signal()